# BPIC17 — data augmentation & sampling

Sits between the Loader and the Training notebook. Reads `Loader/pkl/BPIC_2017_all_5_train.pkl`, applies two transforms, writes the result to `improved/BPIC_2017_all_5_train_augmented.pkl`.

Motivation comes from the C4.5 surrogate-tree analysis on the LSTM:
1. **Per-gateway oversampling** — rare gateways (e.g. `W_Shortened completion`, `W_Assess potential fraud`) collapse the LSTM to a single rule. Balancing windows by SOS event gives those gateways enough gradient signal to be learned.
2. **Static time-feature jitter** — the LSTM surrogate over-relies on `event_elapsed_time_pos_-X` and `seconds_in_day_pos_-X` (noisy continuous features that are easy gradient handles), instead of the activity / Action / EventOrigin features the ground-truth tree uses. Multiplying these features by `(1 + small_noise)` once at preprocessing time forces the optimiser to lean less on the exact magnitude of these signals.

Notes on scope:
- **Train only.** Val and test pickles are untouched (`Loader/pkl/`) so eval metrics remain comparable.
- **Static jitter.** Each augmented row has one perturbed copy of the time features. Dynamic per-epoch jitter would need a custom `Dataset` subclass; that's a follow-up if the static version helps.
- **Existing Loader/Training flow still works** — point `USE_AUGMENTED_TRAIN = False` in the training notebook to use the original `Loader/pkl/` train pickle.

## Imports & paths

In [1]:
import importlib
import sys
from pathlib import Path
from collections import Counter

import numpy as np
import torch

# Notebook lives at: src/interpretability/improved_pipeline/henryk/bpic17/improved/augmentation_and_sampling/<this>.ipynb
sys.path.insert(0, '../../../../../..')  # -> src/

import event_log_loader.new_event_log_loader
importlib.reload(event_log_loader.new_event_log_loader)
from event_log_loader.new_event_log_loader import EventLogDataset

import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

SEED = 17
np.random.seed(SEED)
torch.manual_seed(SEED)

INPUT_TRAIN = Path('../Loader/pkl/BPIC_2017_all_5_train.pkl').resolve()
OUTPUT_TRAIN = Path('../BPIC_2017_all_5_train_augmented.pkl').resolve()
print(f'Reading:  {INPUT_TRAIN}')
print(f'Writing:  {OUTPUT_TRAIN}')

Reading:  /Users/philippeichhorn/IdeaProjects/XAI-Probabilistic_Suffix_Prediction_U-ED-LSTM_pub/src/interpretability/improved_pipeline/henryk/bpic17/improved/Loader/pkl/BPIC_2017_all_5_train.pkl
Writing:  /Users/philippeichhorn/IdeaProjects/XAI-Probabilistic_Suffix_Prediction_U-ED-LSTM_pub/src/interpretability/improved_pipeline/henryk/bpic17/improved/BPIC_2017_all_5_train_augmented.pkl


In [2]:
train_dataset = torch.load(str(INPUT_TRAIN), weights_only=False)
print(f'Loaded train dataset: {len(train_dataset)} windows')

cat_tensors = [t.clone() for t in train_dataset.tensor_list[0]]   # list[Tensor (N, window_size)]
num_tensors = [t.clone() for t in train_dataset.tensor_list[1]]   # list[Tensor (N, window_size)]
case_ids = train_dataset.tensor_list[2].clone() if torch.is_tensor(train_dataset.tensor_list[2]) else list(train_dataset.tensor_list[2])

print('Categorical features:')
for i, (name, size, _) in enumerate(train_dataset.all_categories[0]):
    print(f'  cat[{i:2d}] {name!r:25s} -> {size} classes')
print('Numerical features:')
for i, (name, _, _) in enumerate(train_dataset.all_categories[1]):
    print(f'  num[{i:2d}] {name!r}')

Loaded train dataset: 820824 windows
Categorical features:
  cat[ 0] 'concept:name'            -> 28 classes
  cat[ 1] 'Action'                  -> 7 classes
  cat[ 2] 'org:resource'            -> 151 classes
  cat[ 3] 'EventOrigin'             -> 5 classes
  cat[ 4] 'lifecycle:transition'    -> 9 classes
  cat[ 5] 'case:LoanGoal'           -> 16 classes
  cat[ 6] 'case:ApplicationType'    -> 4 classes
  cat[ 7] 'Accepted'                -> 5 classes
  cat[ 8] 'Selected'                -> 5 classes
Numerical features:
  num[ 0] 'case_elapsed_time'
  num[ 1] 'event_elapsed_time'
  num[ 2] 'day_in_week'
  num[ 3] 'seconds_in_day'
  num[ 4] 'case:RequestedAmount'
  num[ 5] 'FirstWithdrawalAmount'
  num[ 6] 'NumberOfTerms'
  num[ 7] 'MonthlyCost'
  num[ 8] 'CreditScore'


## §A — Per-gateway oversampling

For each window the **gateway** is the SOS event = `activity[-suffix_split-1]` (last position of the prefix the trainer feeds into the decoder).

Strategy: compute the count per gateway. Oversample any gateway whose count is below `target_count = median(counts)` up to `target_count`. Common gateways stay as-is. Rare gateways are capped at `MAX_FACTOR×` original count to avoid blow-up of the rarest classes.

In [3]:
# --- Config ---
SUFFIX_SPLIT = 4
ACTIVITY_CAT_INDEX = 0
MAX_FACTOR = 10                     # don't multiply a gateway's window count more than this
TARGET_QUANTILE = 0.5               # 0.5 = median; raise (e.g. 0.7) to balance more aggressively

activity_label_dict = train_dataset.all_categories[0][ACTIVITY_CAT_INDEX][2]   # name -> idx
idx_to_name = {v: k for k, v in activity_label_dict.items()}
idx_to_name[0] = '<padding>'

activity = cat_tensors[ACTIVITY_CAT_INDEX]
sos_indices = activity[:, -SUFFIX_SPLIT - 1].long()      # (N,)

unique_sos, counts = torch.unique(sos_indices, return_counts=True)
target_count = int(torch.quantile(counts.float(), TARGET_QUANTILE).item())
print(f'Target count per gateway: {target_count} (q={TARGET_QUANTILE})')
print(f'Max factor: {MAX_FACTOR}x original count')
print()
print(f'Per-gateway counts (before oversampling):')
for s, c in sorted(zip(unique_sos.tolist(), counts.tolist()), key=lambda x: -x[1]):
    name = idx_to_name.get(s, f'<idx {s}>')
    print(f'  {c:6d}  {name!r}')

Target count per gateway: 20387 (q=0.5)
Max factor: 10x original count

Per-gateway counts (before oversampling):
  135566  'W_Validate application'
  123937  'W_Call after offers'
  109117  'W_Call incomplete files'
   97051  'W_Complete application'
   40964  'EOS'
   30790  'W_Handle leads'
   27823  'O_Create Offer'
   27823  'O_Created'
   25682  'O_Sent (mail and online)'
   25204  'A_Validating'
   20482  'A_Accepted'
   20482  'A_Concept'
   20482  'A_Create Application'
   20387  'A_Complete'
   15116  'O_Returned'
   14966  'A_Incomplete'
   13498  'O_Cancelled'
   13288  'A_Submitted'
   11162  'A_Pending'
   11162  'O_Accepted'
    6808  'A_Cancelled'
    3052  'O_Refused'
    2451  'A_Denied'
    2054  'W_Assess potential fraud'
    1319  'O_Sent (online only)'
     136  'W_Shortened completion '
      22  'W_Personal Loan collection'


In [4]:
# Compute additional indices to append per under-represented gateway (window-level oversampling)
rng = np.random.RandomState(SEED)
extra_index_lists = []
added_per_gateway = {}
for s, c in zip(unique_sos.tolist(), counts.tolist()):
    if c >= target_count:
        continue
    needed = min(target_count - c, c * (MAX_FACTOR - 1))
    if needed <= 0:
        continue
    matching = (sos_indices == s).nonzero(as_tuple=True)[0].numpy()
    sampled = rng.choice(matching, size=needed, replace=True)
    extra_index_lists.append(torch.from_numpy(sampled).long())
    added_per_gateway[idx_to_name.get(s, f'<idx {s}>')] = needed

if extra_index_lists:
    extra_indices = torch.cat(extra_index_lists)
    print(f'Adding {len(extra_indices)} oversampled rows ({len(extra_index_lists)} gateways)')
    for name, n in sorted(added_per_gateway.items(), key=lambda x: -x[1]):
        print(f'  +{n:5d}  {name!r}')
    cat_tensors = [torch.cat([t, t[extra_indices]]) for t in cat_tensors]
    num_tensors = [torch.cat([t, t[extra_indices]]) for t in num_tensors]
    if torch.is_tensor(case_ids):
        case_ids = torch.cat([case_ids, case_ids[extra_indices]])
    else:
        case_ids = list(case_ids) + [case_ids[i] for i in extra_indices.tolist()]
else:
    print('No oversampling applied (all gateways already meet target).')

print(f'\nDataset size after oversampling: {cat_tensors[0].shape[0]} windows')

Adding 123606 oversampled rows (13 gateways)
  +18333  'W_Assess potential fraud'
  +17936  'A_Denied'
  +17335  'O_Refused'
  +13579  'A_Cancelled'
  +11871  'O_Sent (online only)'
  + 9225  'A_Pending'
  + 9225  'O_Accepted'
  + 7099  'A_Submitted'
  + 6889  'O_Cancelled'
  + 5421  'A_Incomplete'
  + 5271  'O_Returned'
  + 1224  'W_Shortened completion '
  +  198  'W_Personal Loan collection'

Dataset size after oversampling: 944430 windows


## §B — Static time-feature jitter

Multiply selected numerical features by `(1 + uniform(-JITTER_PCT, JITTER_PCT))` per element. Targets `event_elapsed_time` and `seconds_in_day` (the features the LSTM over-relies on per the surrogate tree). Skips `case_elapsed_time` (growing/cumulative — perturbing it would break the monotonicity the trainer expects via `growing_num_values`) and `day_in_week` (effectively discrete: 0–6).

In [5]:
# --- Config ---
JITTER_PCT = 0.05                                   # ±5% multiplicative noise
FEATURES_TO_JITTER = ['event_elapsed_time', 'seconds_in_day']

num_feature_names = [name for name, _, _ in train_dataset.all_categories[1]]
jitter_indices = [i for i, name in enumerate(num_feature_names) if name in FEATURES_TO_JITTER]
skipped = [name for name in num_feature_names if name not in FEATURES_TO_JITTER]

print(f'Jittering {len(jitter_indices)} numerical features at ±{JITTER_PCT*100:.1f}%:')
for i in jitter_indices:
    print(f'  num[{i}] {num_feature_names[i]!r}')
print(f'Skipped (preserved): {skipped}')

rng = torch.Generator().manual_seed(SEED)
for i in jitter_indices:
    noise = (torch.rand(num_tensors[i].shape, generator=rng) * 2 - 1) * JITTER_PCT
    # Preserve zeros (padding); only perturb non-zero values
    mask = (num_tensors[i] != 0).float()
    num_tensors[i] = num_tensors[i] * (1 + noise * mask)

print(f'\nJitter applied. Sample of perturbation magnitude on num[{jitter_indices[0]}]:')
if jitter_indices:
    diff = (num_tensors[jitter_indices[0]] - train_dataset.tensor_list[1][jitter_indices[0]][:cat_tensors[0].shape[0]] if cat_tensors[0].shape[0] <= len(train_dataset) else None)
    # quick stats on a sample
    sample = num_tensors[jitter_indices[0]][:1000].flatten()
    sample_nz = sample[sample != 0]
    if len(sample_nz):
        print(f'  values stats (non-zero, first 1000 rows): min={sample_nz.min():.3f}, max={sample_nz.max():.3f}, mean={sample_nz.mean():.3f}')

Jittering 2 numerical features at ±5.0%:
  num[1] 'event_elapsed_time'
  num[3] 'seconds_in_day'
Skipped (preserved): ['case_elapsed_time', 'day_in_week', 'case:RequestedAmount', 'FirstWithdrawalAmount', 'NumberOfTerms', 'MonthlyCost', 'CreditScore']

Jitter applied. Sample of perturbation magnitude on num[1]:
  values stats (non-zero, first 1000 rows): min=-0.244, max=12.333, mean=-0.054


## Save augmented dataset

Reuses the original dataset's `all_categories` and `encoder_decoder` so the metadata is unchanged. Only the windows have grown (oversampled) and the time features perturbed.

In [6]:
augmented_dataset = EventLogDataset(
    tensor_tuple=(tuple(cat_tensors), tuple(num_tensors), case_ids),
    all_categories=train_dataset.all_categories,
    encoder_decoder=train_dataset.encoder_decoder,
)

OUTPUT_TRAIN.parent.mkdir(parents=True, exist_ok=True)
torch.save(augmented_dataset, str(OUTPUT_TRAIN))
print(f'Saved augmented train pkl to {OUTPUT_TRAIN}')
print(f'  Original windows:  {len(train_dataset)}')
print(f'  Augmented windows: {len(augmented_dataset)}')
print(f'  Growth: {(len(augmented_dataset) / max(len(train_dataset), 1) - 1) * 100:.1f}%')

Saved augmented train pkl to /Users/philippeichhorn/IdeaProjects/XAI-Probabilistic_Suffix_Prediction_U-ED-LSTM_pub/src/interpretability/improved_pipeline/henryk/bpic17/improved/BPIC_2017_all_5_train_augmented.pkl
  Original windows:  820824
  Augmented windows: 944430
  Growth: 15.1%


### To use this in training

Open `improved/Training/notebook/full_enc_dec_lstm_gn.ipynb` and set:

```python
USE_AUGMENTED_TRAIN = True
```

When the flag is on, the training notebook reads the augmented train pickle from `improved/BPIC_2017_all_5_train_augmented.pkl` instead of `Loader/pkl/BPIC_2017_all_5_train.pkl`. Val and test still come from `Loader/pkl/`. Set the flag back to `False` to recover the original behaviour.